# `generate_number_labyrinth_board` — Unit Tests

This notebook tests the board-generation function in isolation.
All helper functions are copied from `full_board_generation.ipynb` with the
uniqueness constraint removed for simplicity.

**Tests cover:**
- Structural completeness (all keys present, board size, path endpoints)
- Value ranges (global min/max bounds)
- Path correctness (strictly increasing, steps in range, adjacency, length bounds)
- Movement rule (the solution path is a valid walk under the game rule)
- Reproducibility (same seed → identical board)
- Reliability (success rate across N runs per level config)

In [1]:
import random
import math
from collections import deque

# ── Global defaults (used as fallbacks inside helper functions) ────────────────
MIN_STEP = 1
MAX_STEP = 20
BASE_MIN = 1
BASE_MAX = 20
P_HIGHER = 0.5

DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

In [ ]:
# ── generate_solution_path_dfs ─────────────────────────────────────────────────
def generate_solution_path_dfs(cols, rows, min_length=None, max_length=None, rng=None):
    if rng is None:
        rng = random

    start = (0, 0)
    goal  = (cols - 1, rows - 1)

    def neighbors(c, r):
        return [(c+dc, r+dr) for dc, dr in DIRS
                if 0 <= c+dc < cols and 0 <= r+dr < rows]

    def manhattan(c, r):
        return abs(goal[0]-c) + abs(goal[1]-r)

    stack = [(start, frozenset([start]), (start,))]
    while stack:
        pos, visited, path = stack.pop()
        cands = list(neighbors(*pos))
        rng.shuffle(cands)
        for nc in cands:
            if nc in visited:
                continue
            if any(nb in visited and nb != pos for nb in neighbors(*nc)):
                continue
            new_len = len(path) + 1
            if max_length is not None and new_len + manhattan(*nc) > max_length:
                continue
            new_path    = path + (nc,)
            new_visited = visited | {nc}
            if nc == goal:
                if min_length is None or new_len >= min_length:
                    return new_path
            else:
                stack.append((nc, new_visited, new_path))
    return None


# ── assign_path_values ─────────────────────────────────────────────────────────
def assign_path_values(path, min_step=None, max_step=None, base_val=None, rng=None):
    if min_step is None: min_step = MIN_STEP
    if max_step is None: max_step = MAX_STEP
    if rng      is None: rng      = random
    if base_val is None: base_val = rng.randint(BASE_MIN, BASE_MAX)

    values  = {}
    current = base_val
    for cell in path:
        values[cell] = current
        current += rng.randint(min_step, max_step)
    return values


# ── fill_board ─────────────────────────────────────────────────────────────────
def fill_board(
    cols, rows, path, path_values,
    p_higher   = None,
    min_step   = None,
    max_step   = None,
    global_min = 1,
    global_max = 999,
    rng        = None,
):
    if p_higher is None: p_higher = P_HIGHER
    if min_step is None: min_step = MIN_STEP
    if max_step is None: max_step = MAX_STEP
    if rng      is None: rng      = random

    board  = dict(path_values)
    labels = {cell: 'A' for cell in path}

    def adj(c, r):
        return [(c+dc, r+dr) for dc, dr in DIRS
                if 0 <= c+dc < cols and 0 <= r+dr < rows]

    def sample_step():
        return rng.randint(min_step, max_step)

    def clamp(v):
        return max(global_min, min(global_max, v))

    while len(board) < cols * rows:
        snapshot = set(board.keys())
        neighbor_batch = {
            nc
            for cell in snapshot
            for nc in adj(*cell)
            if nc not in snapshot
        }
        if not neighbor_batch:
            break

        for cell in neighbor_batch:
            numbered_nbrs = [n for n in adj(*cell) if n in snapshot]
            n_nbrs = len(numbered_nbrs)

            if n_nbrs == 1:
                nbr_val = board[numbered_nbrs[0]]
                if rng.random() < p_higher:
                    new_val, new_label = clamp(nbr_val + sample_step()), 'A'
                else:
                    new_val, new_label = clamp(nbr_val - sample_step()), 'B'

            elif n_nbrs == 2:
                v0, v1 = board[numbered_nbrs[0]], board[numbered_nbrs[1]]
                l0, l1 = labels[numbered_nbrs[0]], labels[numbered_nbrs[1]]
                lo, hi = min(v0, v1), max(v0, v1)
                if l0 == 'A' and l1 == 'A':
                    new_val, new_label = clamp(hi + sample_step()), 'B'
                else:
                    if hi - lo >= 2:
                        new_val, new_label = rng.randint(lo + 1, hi - 1), 'A'
                    elif clamp(lo - sample_step()) < lo:
                        new_val, new_label = clamp(lo - sample_step()), 'B'
                    else:
                        new_val, new_label = clamp(hi + sample_step()), 'A'

            else:
                min_adj = min(board[n] for n in numbered_nbrs)
                new_val, new_label = clamp(min_adj - sample_step()), 'B'

            board[cell]  = new_val
            labels[cell] = new_label

    return board, labels


# ── generate_number_labyrinth_board ────────────────────────────────────────────
def generate_number_labyrinth_board(
    cols,
    rows,
    min_pct       = 0.35,
    max_pct       = 0.45,
    p_higher      = 0.75,
    path_min_step = 3,
    path_max_step = 5,
    fill_min_step = 1,
    fill_max_step = 25,
    base_val      = None,
    global_min    = 1,
    global_max    = 999,
    max_retries   = 20,
    rng           = None,
):
    if rng is None:
        rng = random.Random()

    T  = cols * rows
    lo = math.floor(min_pct * T)
    hi = math.floor(max_pct * T)

    for _ in range(max_retries):
        attempt_rng = random.Random(rng.randint(0, 999_999))

        path = generate_solution_path_dfs(cols, rows, min_length=lo, max_length=hi, rng=attempt_rng)
        if path is None:
            continue

        path_values = assign_path_values(
            path,
            min_step = path_min_step,
            max_step = path_max_step,
            base_val = base_val,
            rng      = attempt_rng,
        )

        board, labels = fill_board(
            cols, rows, path, path_values,
            p_higher   = p_higher,
            min_step   = fill_min_step,
            max_step   = fill_max_step,
            global_min = global_min,
            global_max = global_max,
            rng        = attempt_rng,
        )

        return dict(
            grid = [[board[(c, r)] for c in range(cols)] for r in range(rows)],
            path = [{'col': c, 'row': r} for c, r in path],
        )

    return None


print('All functions loaded.')

All functions loaded.


In [ ]:
# ── Level configurations ───────────────────────────────────────────────────────
# Each entry maps directly to generate_number_labyrinth_board kwargs.
# min_pct / max_pct are derived from path_min/max divided by total cells
# and kept as explicit fractions for clarity; they are recomputed per call.

LEVEL_CONFIGS = {
    1: dict(
        cols=4, rows=4,
        path_min_step=3,  path_max_step=10,
        fill_min_step=1,  fill_max_step=7,
        min_pct=0.35,     max_pct=0.45,
        global_min=1,     global_max=100,
        p_higher=0.75,
    ),
    2: dict(
        cols=5, rows=5,
        path_min_step=4,  path_max_step=14,
        fill_min_step=1,  fill_max_step=10,
        min_pct=0.35,     max_pct=0.45,
        global_min=1,     global_max=150,
        p_higher=0.75,
    ),
    3: dict(
        cols=6, rows=6,
        path_min_step=5,  path_max_step=20,
        fill_min_step=1,  fill_max_step=20,
        min_pct=0.35,     max_pct=0.45,
        global_min=1,     global_max=350,
        p_higher=0.75,
    ),
    4: dict(
        cols=7, rows=7,
        path_min_step=6,  path_max_step=25,
        fill_min_step=1,  fill_max_step=30,
        min_pct=0.35,     max_pct=0.45,
        global_min=1,     global_max=700,
        p_higher=0.75,
    ),
    5: dict(
        cols=8, rows=8,
        path_min_step=7,  path_max_step=30,
        fill_min_step=1,  fill_max_step=40,
        min_pct=0.35,     max_pct=0.45,
        global_min=1,     global_max=999,
        p_higher=0.75,
    ),
}

print('Level configs:')
for lvl, cfg in LEVEL_CONFIGS.items():
    T = cfg['cols'] * cfg['rows']
    lo = math.floor(cfg['min_pct'] * T)
    hi = math.floor(cfg['max_pct'] * T)
    print(f"  Level {lvl}: {cfg['cols']}x{cfg['rows']}  "
          f"path_step=[{cfg['path_min_step']},{cfg['path_max_step']}]  "
          f"fill_step=[{cfg['fill_min_step']},{cfg['fill_max_step']}]  "
          f"path_len=[{lo},{hi}]")

Level configs:
  Level 1: 4x4  path_step=[3,10]  fill_step=[1,7]  path_len=[5,8]
  Level 2: 5x5  path_step=[4,14]  fill_step=[1,10]  path_len=[8,13]
  Level 3: 6x6  path_step=[5,20]  fill_step=[1,20]  path_len=[12,19]
  Level 4: 7x7  path_step=[6,25]  fill_step=[1,30]  path_len=[17,26]
  Level 5: 8x8  path_step=[7,30]  fill_step=[1,40]  path_len=[22,35]


---
## Unit tests

Each test function receives a result dict (the output of `generate_number_labyrinth_board`)  
and a config dict, and returns `(passed: bool, message: str)`.  
The runner at the bottom calls all tests across all level configs and prints a summary.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Individual test functions
# Each returns (passed: bool, detail: str)
# ─────────────────────────────────────────────────────────────────────────────

def t01_required_keys(result, cfg):
    """Result dict contains all required keys."""
    required = {'grid', 'path'}
    missing  = required - set(result.keys())
    if missing:
        return False, f'missing keys: {missing}'
    return True, 'all keys present'


def t02_board_size(result, cfg):
    """Grid has exactly rows rows, each with cols values."""
    cols, rows = cfg['cols'], cfg['rows']
    grid = result['grid']
    if len(grid) != rows:
        return False, f'expected {rows} rows, got {len(grid)}'
    bad_rows = [r for r, row in enumerate(grid) if len(row) != cols]
    if bad_rows:
        return False, f'rows {bad_rows[:5]} have wrong column count (expected {cols})'
    return True, f'{rows}×{cols} = {rows * cols} cells'


def t03_path_endpoints(result, cfg):
    """Path starts at col=0,row=0 and ends at col=cols-1,row=rows-1."""
    cols, rows = cfg['cols'], cfg['rows']
    path = result['path']
    start = path[0]
    if start['col'] != 0 or start['row'] != 0:
        return False, f'path starts at {start}, expected col=0,row=0'
    goal = path[-1]
    if goal['col'] != cols - 1 or goal['row'] != rows - 1:
        return False, f'path ends at {goal}, expected col={cols-1},row={rows-1}'
    return True, f'start=(0,0) goal=({cols-1},{rows-1}) ✓'


def t04_path_adjacency(result, cfg):
    """Every consecutive pair of path cells is adjacent (Manhattan distance == 1)."""
    path = result['path']
    bad  = []
    for i in range(len(path) - 1):
        c1, r1 = path[i]['col'],   path[i]['row']
        c2, r2 = path[i+1]['col'], path[i+1]['row']
        if abs(c1 - c2) + abs(r1 - r2) != 1:
            bad.append((path[i], path[i + 1]))
    if bad:
        return False, f'{len(bad)} non-adjacent steps: {bad[:3]}'
    return True, 'all path steps are adjacent'


def t05_path_length_in_range(result, cfg):
    """Path length is within [floor(min_pct*T), floor(max_pct*T)]."""
    T    = cfg['cols'] * cfg['rows']
    lo   = math.floor(cfg['min_pct'] * T)
    hi   = math.floor(cfg['max_pct'] * T)
    plen = len(result['path'])
    if not (lo <= plen <= hi):
        return False, f'path length {plen} not in [{lo}, {hi}]'
    return True, f'path length {plen} in [{lo}, {hi}]'


def t06_path_values_strictly_increasing(result, cfg):
    """Values along the solution path are strictly increasing."""
    path = result['path']
    grid = result['grid']
    seq  = [grid[step['row']][step['col']] for step in path]
    violations = [(i, seq[i], seq[i+1]) for i in range(len(seq)-1) if seq[i] >= seq[i+1]]
    if violations:
        return False, f'{len(violations)} non-increasing steps: {violations[:3]}'
    return True, f'path values strictly increasing [{seq[0]}..{seq[-1]}]'


def t07_path_step_sizes_in_range(result, cfg):
    """Each step between consecutive path values is in [path_min_step, path_max_step]."""
    path = result['path']
    grid = result['grid']
    lo   = cfg['path_min_step']
    hi   = cfg['path_max_step']
    seq  = [grid[step['row']][step['col']] for step in path]
    steps = [seq[i+1] - seq[i] for i in range(len(seq)-1)]
    bad   = [s for s in steps if not (lo <= s <= hi)]
    if bad:
        return False, f'{len(bad)} steps out of [{lo},{hi}]: {bad[:5]}'
    return True, f'all {len(steps)} path steps in [{lo},{hi}]'


def t08_global_value_bounds(result, cfg):
    """All grid values are within [global_min, global_max]."""
    lo   = cfg['global_min']
    hi   = cfg['global_max']
    grid = result['grid']
    bad  = [(r, c, grid[r][c])
            for r, row in enumerate(grid)
            for c, v in enumerate(row)
            if not (lo <= v <= hi)]
    if bad:
        return False, f'{len(bad)} values outside [{lo},{hi}]: {bad[:5]}'
    return True, f'all values in [{lo},{hi}]'


def t09_movement_rule_on_path(result, cfg):
    """Each path step goes to a strictly higher-valued adjacent cell (game movement rule)."""
    path = result['path']
    grid = result['grid']
    violations = []
    for i in range(len(path) - 1):
        v1 = grid[path[i]['row']][path[i]['col']]
        v2 = grid[path[i+1]['row']][path[i+1]['col']]
        if v1 >= v2:
            violations.append((path[i], v1, path[i+1], v2))
    if violations:
        return False, f'{len(violations)} movement-rule violations: {violations[:3]}'
    return True, 'movement rule satisfied on entire path'


def t10_path_no_self_intersect(result, cfg):
    """Path visits no cell more than once."""
    path   = result['path']
    tuples = [(s['col'], s['row']) for s in path]
    if len(tuples) != len(set(tuples)):
        from collections import Counter
        dupes = [c for c, n in Counter(tuples).items() if n > 1]
        return False, f'path self-intersects at {dupes}'
    return True, 'path has no self-intersections'


def t11_reproducibility(result, cfg):
    """Same seed produces an identical board on a second call."""
    seed = random.randint(0, 999_999)
    r1   = generate_number_labyrinth_board(**cfg, rng=random.Random(seed))
    r2   = generate_number_labyrinth_board(**cfg, rng=random.Random(seed))
    if r1 is None or r2 is None:
        return None, 'skipped — generation failed (try another seed)'
    if r1['grid'] != r2['grid']:
        return False, 'grids differ with same seed'
    if r1['path'] != r2['path']:
        return False, 'paths differ with same seed'
    return True, f'seed={seed} gives identical boards'


def t12_reliability(result, cfg, n=50, max_fail_rate=0.05):
    """Function succeeds on >= 95% of n consecutive calls."""
    n_ok = sum(
        1 for _ in range(n)
        if generate_number_labyrinth_board(**cfg) is not None
    )
    rate = n_ok / n
    if rate < (1 - max_fail_rate):
        return False, f'success rate {rate:.0%} < {1-max_fail_rate:.0%} ({n_ok}/{n} ok)'
    return True, f'success rate {rate:.0%} ({n_ok}/{n} ok)'


# ── Test registry ──────────────────────────────────────────────────────────────
RESULT_TESTS = [
    t01_required_keys,
    t02_board_size,
    t03_path_endpoints,
    t04_path_adjacency,
    t05_path_length_in_range,
    t06_path_values_strictly_increasing,
    t07_path_step_sizes_in_range,
    t08_global_value_bounds,
    t09_movement_rule_on_path,
    t10_path_no_self_intersect,
]

SELF_CONTAINED_TESTS = [
    t11_reproducibility,
    t12_reliability,
]

print(f'{len(RESULT_TESTS) + len(SELF_CONTAINED_TESTS)} tests defined.')

12 tests defined.


In [5]:
# ── Test runner ────────────────────────────────────────────────────────────────
SEED = 42   # fixed seed for the shared result used by RESULT_TESTS

total_pass = total_fail = total_skip = 0
all_failures = []

for level, cfg in LEVEL_CONFIGS.items():
    print(f"\n{'━'*64}")
    print(f"  Level {level} — {cfg['cols']}×{cfg['rows']}  "
          f"path_step=[{cfg['path_min_step']},{cfg['path_max_step']}]  "
          f"fill_step=[{cfg['fill_min_step']},{cfg['fill_max_step']}]")
    print(f"{'━'*64}")

    # Generate one shared result for structural tests
    shared_result = generate_number_labyrinth_board(**cfg, rng=random.Random(SEED))
    if shared_result is None:
        print(f"  [WARN] seed={SEED} produced None — skipping result-based tests")
        shared_result = {}

    for fn in RESULT_TESTS:
        if not shared_result:
            status, detail = None, 'skipped — no result'
        else:
            try:
                status, detail = fn(shared_result, cfg)
            except Exception as e:
                status, detail = False, f'raised {type(e).__name__}: {e}'

        if status is True:
            icon = '✓'; total_pass += 1
        elif status is False:
            icon = '✗'; total_fail += 1
            all_failures.append((level, fn.__name__, detail))
        else:
            icon = '–'; total_skip += 1

        print(f"  {icon}  {fn.__name__:<42}  {detail}")

    for fn in SELF_CONTAINED_TESTS:
        try:
            status, detail = fn(shared_result, cfg)
        except Exception as e:
            status, detail = False, f'raised {type(e).__name__}: {e}'

        if status is True:
            icon = '✓'; total_pass += 1
        elif status is False:
            icon = '✗'; total_fail += 1
            all_failures.append((level, fn.__name__, detail))
        else:
            icon = '–'; total_skip += 1

        print(f"  {icon}  {fn.__name__:<42}  {detail}")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*64}")
print(f"  RESULTS:  {total_pass} passed   {total_fail} failed   {total_skip} skipped")
print(f"{'='*64}")
if all_failures:
    print("\n  Failures:")
    for level, name, detail in all_failures:
        print(f"    Level {level}  {name}: {detail}")
else:
    print("  All tests passed ✓")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Level 1 — 4×4  path_step=[3,10]  fill_step=[1,7]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✓  t01_required_keys                           all keys present
  ✓  t02_board_size                              4×4 = 16 cells
  ✓  t03_path_endpoints                          start=(0,0) goal=(3,3) ✓
  ✓  t04_path_adjacency                          all path steps are adjacent
  ✓  t05_path_length_in_range                    path length 7 in [5, 8]
  ✓  t06_path_values_strictly_increasing         path values strictly increasing [15..52]
  ✓  t07_path_step_sizes_in_range                all 6 path steps in [3,10]
  ✓  t08_global_value_bounds                     all values in [1,999]
  ✓  t09_movement_rule_on_path                   movement rule satisfied on entire path
  ✓  t10_path_no_self_intersect                  path has no self-intersections
  ✓  t11_reproducibility                         seed=78217